# 01 · Suppression-mechanism confirmation  —  `DIFF@pivot = ATTN(q = −PROGRESS, k = TASK)`

Hypothesis: at the pivot the goal heads compute the leftover task by a single attention op whose
**query is a function of PROGRESS** (the done-set) and whose **keys/values are the TASK spec**. The
query *suppresses* attention onto the task items already satisfied by PROGRESS; the surviving mass is
the frontier item, written by the OV.

Battery (one 3-predicate example, 2–3 instances — flow-polishing only, no aggregate stats):
- **A1** graded suppression + OV-write norm — does attention onto item-k collapse exactly when PROGRESS completes k?
- **A2** anti-alignment — is the query update from completing item i anti-parallel to item-i's task keys?
- **A3** sign necessity — reflecting the done-update through baseline should *not* skip to the frontier.
- **B1** query linearity (geometric) — `Δq({a,b}) ≈ Δq({a}) + Δq({b})`?
- **B2** compositional causal injection — inject `q({a})+q({b})−q({})`; does it *generate* item c only? saturation → boundary.

### 🔧 Intervention mechanism (reference) — pre-RoPE query transplant

*The same machinery underlies Spec 1 (`00_diff.ipynb`), A3, and B2. Read this once; every
`inject`-style arm below is a variation on it.* Numbers below are for **Qwen2.5-Coder-32B**
(40 q-heads, 8 kv-heads, `head_dim=128` → `q_proj` output width = 40×128 = 5120, `GROUP = 40/8 = 5`).

#### 1 · Source / destination prompts + pivots

Both prompts are the **same TASK**, differing only in how much of the prefill (**PROGRESS**) is
written. The task description precedes the prefill, so its tokens sit at **identical positions** in both
→ their **keys/values match and are never touched**. Only the **pivot query** differs.

```
                    ┌────────────── SHARED: TASK spec  (KEYS / VALUES live here) ──────────────┐
 SOURCE (pab)       │ Write func(words) that computes:                                          │
 "a,b done"         │  (1) `a`: words longer than 3 chars                                        │
                    │  (2) `b`: count of palindromic words                                       │
                    │  (3) `c`: length of the longest word                                       │
                    └───────────────────────────────────────────────────────────────────────────┘
                    <code>
                    def func(words: list[str]):
                        a = [w for w in words if len(w) > 3]   ┐ PROGRESS = {a,b}
                        b = sum(w == w[::-1] for w in words)   ┘ (this region DIFFERS)
                        ‸   ← PIVOT_SRC   (trailing indent = last token; about to write next var)

 DEST (p0)          ┌────────────── SAME TASK, SAME TOKEN POSITIONS ───────────────────────────┐
 "nothing done"     │ ... identical (1)(2)(3) ...                                                │
                    └───────────────────────────────────────────────────────────────────────────┘
                    <code>
                    def func(words: list[str]):
                        ‸   ← PIVOT_DEST   PROGRESS = {}
```

- **Pivot = last token of the prompt** (`pivot_of(p) = input_ids.shape[1] - 1`).
- SRC's pivot has a *larger absolute index* (extra a,b lines) — harmless: the query is cached
  **pre-RoPE** (position-independent) and re-rotated at DEST's own position on injection.

#### 2 · What is cached from the SRC pivot

```python
qab = cache_src_proj(pab, [pivot_of(pab)], Q_LAYERS, "q")   # {layer: (1, 5120)}
```

- Traces each goal-layer's `…self_attn.q_proj` **output** at the **pivot position only**.
- It is the **`q_proj` output, PRE-RoPE** (RoPE is applied later, inside attention).
- The full 5120-wide row is stored, but **only the goal-head 128-wide slices are used** on injection.
- `q_reflect` has no source prompt — it is *synthesized*: `q_lincomb([(2.0, q0), (-1.0, qab)])` = `2·q0 − qab`.

#### 3 · The injection — overwrite only goal-head q-slices at the pivot

`proj_patch_hooks` registers a forward hook on each `q_proj`; for each goal head `h` and target
position `p` it does `o[:, p, h*128:(h+1)*128] = qab[layer][row, h*128:(h+1)*128]`. All other heads
keep DEST's own query; **keys/values (TASK) are untouched**. RoPE rotates the injected vector at `p`.

```
   DEST q_proj output @ PIVOT_DEST, layer L:   [ h0 | h1 |███| h3 | … |███| … | h39 ]
                                                          ▲            ▲
                                                    goal heads overwritten with qab's slices
                                                    (███), everything else = DEST's own query
```

**Two phases (`greedy`):**

```
   PREFILL  ...task...  def func(...):  [PIVOT_DEST]‸    ← prefill_specs (positions=[PIVOT_DEST])
            → the pivot's patched attention leaves a modified residual + K/V in the KV cache
              (this one-shot footprint "rides the cache"; the QUERY itself is never cached)

   DECODE 1     [tok₁]‸    ← decode_specs (positions=[-1]): patch the NEW token's own query
   DECODE 2     [tok₂]‸    ← same fixed qab again  (queries are recomputed fresh every step)
   ...                        repeated for the first n_steer steps, then released
```

- **Queries are never stored in the KV cache — only K and V are.** So the prefill patch's persistence
  is only the pivot's *downstream K/V footprint*; to actually **hold** the steer you must re-inject the
  query every decode step (`decode_specs`, `positions=[-1]`).
- Each step stamps the **same fixed pre-RoPE `qab`** onto an ordinary (non-pivot) generated token,
  overwriting the query it would have computed. This **freezes PROGRESS** → run forever it would loop;
  `n_steer` caps it (e.g. 20) so the frontier decision carries through without runaway.

> **vs Spec 1 (`00_diff`):** identical mechanism (pre-RoPE `q_proj`-output transplant, re-clamped per
> step). Differences are incidental: head set (`UNION_HEADS` there vs `GOAL_HEADS` here), readout
> (attention-delta/goalscope vs greedy generation), and A3's synthetic `q_reflect` control.


In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")            # so `_diff_primitives` is importable
import torch, numpy as np
import _diff_primitives as P
P.setup("Qwen/Qwen2.5-Coder-32B-Instruct")   # loads model + head masks (once)
from _diff_primitives import *                # helper functions into scope
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP

2026-08-29 22:43:21 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


In [2]:
# ── intervention head set: which heads receive the pre-RoPE query transplant (A3 · B2 · B2b) ──
# Flip this to change the causal payload. Options: GOAL_HEADS · BOUNDARY_HEADS · UNION_HEADS
# (= goal ∪ boundary). Custom compose: sorted(set(GOAL_HEADS) | set(BOUNDARY_HEADS)).
# NOTE: readouts A1/A2/B1 still use GOAL_HEADS directly — this global governs interventions only.
PATCH_HEADS = GOAL_HEADS
print(f"PATCH_HEADS = {len(PATCH_HEADS)} heads over layers {sorted({l for l, _ in PATCH_HEADS})}")


PATCH_HEADS = 122 heads over layers [28, 29, 30, 31, 32, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55]


In [3]:
# ── multipredicate example: 4 INDEPENDENT computations, one assignment line each ──
# All orderings valid — the pivot choice is a genuine "which sub-task next", not a data dependency.
# Item `a` is the ANCHOR: ALWAYS prefilled — one in-context example that fixes the coding style
# (an inline list-comprehension). Progress analysis focuses on the remaining {b, c, d}.

USE_NATURAL_NAMES = True        # False → slot names a/b/c/d ; True → descriptive variable names

# slot: (condition text, RHS expression, natural variable name)
ITEMS = {
    "a": ("the list of words that start with 'a'",         "[w for w in words if w.startswith('a')]", "a_words"),
    "b": ("the list of words with more than five letters", "[w for w in words if len(w) > 5]",        "long_words"),
    "c": ("the length of the longest word",                "max(len(w) for w in words)",              "longest_len"),
    "d": ("the count of palindromic words",                "sum(w == w[::-1] for w in words)",        "palindrome_count"),

}
ORDER   = ["a", "b", "c", "d"]
ANCHOR  = "a"                    # always prefilled, never a pivot choice
ANALYZE = ["b", "c", "d"]       # the progress set we sweep / inject over

def varname(k): return ITEMS[k][2] if USE_NATURAL_NAMES else k
def assign(k):  return f"{varname(k)} = {ITEMS[k][1]}"

# conditions only — no dictated variable names; the LM names what it writes at/after the pivot
TASK_SPEC = "\n".join(f"({i+1}) {ITEMS[k][0]}" for i, k in enumerate(ORDER))
QUESTION = ("Write a Python program that implements a function `func` which, given a list of "
            "strings `words`, computes:\n" + TASK_SPEC + "\n")

def build_prefill(done, extra=""):
    """Prefill = ANCHOR (always) + assignments in `done` (⊆ ANALYZE), kept in ORDER; pivot at next indent."""
    keep = [k for k in ORDER if k == ANCHOR or k in done]
    lines = ["<code>", "def func(words: list[str]):"] + [f"    {assign(k)}" for k in keep]
    return "\n".join(lines) + "\n    " + extra   # trailing indent = pivot (about to write next var)

def build_prompt(done, extra=""):
    return get_prompt(QUESTION, build_prefill(done, extra))

def pivot_of(prompt):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] - 1

# task-item token spans are identical across done-sets (task precedes prefill) — compute once.
_p0 = build_prompt([])                        # {} means "anchor a only"
ITEM_SPAN = {k: token_span(_p0, ITEMS[k][0]) for k in ORDER}
show_prompt("done={a} (anchor only)", _p0,               ITEM_SPAN["b"])
show_prompt("done={a,b}           ", build_prompt(["b"]), ITEM_SPAN["c"])
print("item spans:", ITEM_SPAN, " pivot({a})=", pivot_token(_p0))

╭─ done={a} (anchor only)  TASK span (79, 88)  ·  pivot=['    ']  ·  L=143 ───────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that start with 'a'\n                                                                     │
│ (2) the list of words with more than five letters\n                                                             │
│ (3) the length of the longest word\n                                                                            │
│ (4) the count of palindromic words\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a_words = [w for w in words if w.startswith('a')]\n                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ done={a,b}             TASK span (92, 98)  ·  pivot=['    ']  ·  L=161 ────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that start with 'a'\n                                                                     │
│ (2) the list of words with more than five letters\n                                                             │
│ (3) the length of the longest word\n                                                                            │
│ (4) the count of palindromic words\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a_words = [w for w in words if w.startswith('a')]\n                                                         │
│     long_words = [w for w in words if len(w) > 5]\n                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

item spans: {'a': (66, 76), 'b': (79, 88), 'c': (92, 98), 'd': (102, 110)}  pivot({a})= ['    ']


## A1 · Graded suppression + OV-write norm

Anchor `a` is always prefilled. Sweep PROGRESS ∈ {{a}, {a,b}, {a,b,c}}. For each: value-weighted
attention from the pivot onto each analyzed item's spec tokens (mean over GOAL heads), and the norm of
the aggregated goal-head write.
**Predict:** attention onto item-k drops sharply when k enters PROGRESS; frontier item retains mass;
write-norm decreases monotonically toward the all-done floor.

In [4]:
from src.attention import get_attention_matrices

DONE_LEVELS = [[], ["b"], ["b", "c"]]         # PROGRESS beyond the anchor: {a}, {a,b}, {a,b,c}
tbl = Table(title="A1 · pivot→TASK-item attention (mean over GOAL heads)  +  ‖goal write‖", highlight=False)
tbl.add_column("PROGRESS")
for k in ANALYZE: tbl.add_column(f"attn·{k}", justify="right")
tbl.add_column("‖write‖", justify="right")
for done in DONE_LEVELS:
    p = build_prompt(done)
    piv = pivot_of(p)
    tok = prepare_input(prompts=p, tokenizer=mt)
    info = get_attention_matrices(tok, mt, value_weighted=True)
    row = ["{" + ",".join([ANCHOR] + done) + "}"]
    for k in ANALYZE:
        a = pivot_attn(info, GOAL_HEADS, list(range(*ITEM_SPAN[k])), piv).sum()
        mark = "[green]" if k not in done else "[red]"
        row.append(f"{mark}{a:.3f}[/]")
    row.append(f"{write_norm(p, GOAL_HEADS, piv):.2f}")
    tbl.add_row(*row)
rich_print(tbl)
rich_print("[grey58]green = still-undone item (expect mass), red = done item (expect suppressed); anchor a always done[/grey58]")

 A1 · pivot→TASK-item attention (mean over GOAL  
             heads)  +  ‖goal write‖             
┏━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ PROGRESS ┃ attn·b ┃ attn·c ┃ attn·d ┃ ‖write‖ ┃
┡━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ {a}      │  2.897 │  0.517 │  0.377 │  240.91 │
│ {a,b}    │  0.259 │  2.205 │  0.331 │  213.62 │
│ {a,b,c}  │  0.153 │  0.181 │  2.542 │  214.65 │
└──────────┴────────┴────────┴────────┴─────────┘

green = still-undone item (expect mass), red = done item (expect suppressed); anchor a always done

## A2 · Anti-alignment of the query update with the completed item's keys

Baseline is the anchor `{a}`. `q` is pre-RoPE at the pivot; `k` is pre-RoPE at item-b's spec tokens.
For every GOAL q-head, cosine of `Δq = q({a,b}) − q({a})` (q-head slice) against the mean item-b key
(its kv-head slice, GQA `h//GROUP`).
**Predict:** cosine < 0 (the completed item is *subtracted into* the query, suppressing its keys).

In [5]:
qheads = heads_per_layer(GOAL_HEADS, "q")
# cache band must cover BOTH the readout heads (GOAL) and the intervention heads (PATCH_HEADS),
# else proj_patch_hooks hits a layer the caches lack (KeyError) when PATCH_HEADS ⊈ GOAL band.
Q_LAYERS = sorted({l for l, _ in GOAL_HEADS} | {l for l, _ in PATCH_HEADS})
p0, pb = build_prompt([]), build_prompt(["b"])       # baseline = anchor {a}; complete item b
q0 = cache_src_proj(p0, [pivot_of(p0)], Q_LAYERS, "q")
qb = cache_src_proj(pb, [pivot_of(pb)], Q_LAYERS, "q")
dq = q_lincomb([(1.0, qb), (-1.0, q0)])              # done-update from completing b: q({a,b}) − q({a})

# mean item-b KEY at its spec tokens (positions identical across prompts; use p0)
kb = cache_src_proj(p0, list(range(*ITEM_SPAN["b"])), Q_LAYERS, "k")
kbar = {l: kb[l].mean(0, keepdim=True) for l in Q_LAYERS}     # {layer: (1, width)}

hd = mt.head_dim
coss = []
for (l, h) in GOAL_HEADS:
    if l not in dq: continue
    qv = dq[l][0, h * hd:(h + 1) * hd]
    kvh = h // GROUP
    kv = kbar[l][0, kvh * hd:(kvh + 1) * hd]
    coss.append(cos(qv, kv))
coss = np.array(coss)
print(f"Δq·k(item b) cosine over {len(coss)} GOAL q-heads: "
      f"mean={coss.mean():+.3f}  median={np.median(coss):+.3f}  frac<0={ (coss < 0).mean():.2f}")
print("(strongly negative mean ⇒ completing b subtracts b's key-direction into the query — suppression)")

Δq·k(item b) cosine over 122 GOAL q-heads: mean=-0.165  median=-0.160  frac<0=0.89
(strongly negative mean ⇒ completing b subtracts b's key-direction into the query — suppression)


## A3 · Sign necessity — reflect the done-update

If the *sign* of the query update selects, then reflecting the done-update through baseline —
`q_reflect = q({a}) − (q({a,b,c}) − q({a})) = 2·q({a}) − q({a,b,c})` — should **not** behave as if b,c
are done (should not skip to d), unlike the true `q({a,b,c})`. Injected into the `{a}`-baseline dest and generated.

In [6]:
# ── A3: reflect the done-update over {b,c}; sign necessity (see reference cell) ──
pbc = build_prompt(["b", "c"])                       # {a,b,c} done  (anchor + b + c)
qbc = cache_src_proj(pbc, [pivot_of(pbc)], Q_LAYERS, "q")
q_reflect = q_lincomb([(2.0, q0), (-1.0, qbc)])      # 2·q({a}) − q({a,b,c})  = reflect the done-update

def q_inject_specs(qcache, dest_pivot):
    qh = heads_per_layer(PATCH_HEADS, "q")
    return ([dict(which="q", cache=qcache, heads=qh, positions=[dest_pivot])],
            [dict(which="q", cache=qcache, heads=qh, positions=[-1])])

dest_tok = prepare_input(prompts=p0, tokenizer=mt)   # DEST = baseline {a}
piv0 = pivot_of(p0)
pf_bc, dc_bc = q_inject_specs(qbc,       piv0)
pf_rf, dc_rf = q_inject_specs(q_reflect, piv0)

show_prompt("SRC  = {a,b,c} done", pbc, ITEM_SPAN["d"])   # pivot shown in panel title
show_prompt("DEST = {a} (anchor)", p0,  ITEM_SPAN["b"])

rich_print(f"[grey70]clean {{a}} (expect writes b) :[/grey70] \n{escape(greedy(dest_tok, max_new=32))}")
rich_print(f"[sea_green3]inject q(b,c) (expect d)   :[/sea_green3] "
           f"\n{escape(greedy(dest_tok, prefill_specs=pf_bc, decode_specs=dc_bc, n_steer=40, max_new=64))}")
rich_print(f"[salmon1]inject q_reflect (expect ≠d) :[/salmon1] "
           f"\n{escape(greedy(dest_tok, prefill_specs=pf_rf, decode_specs=dc_rf, n_steer=40, max_new=64))}")

╭─ SRC  = {a,b,c} done  TASK span (102, 110)  ·  pivot=['    ']  ·  L=174 ────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that start with 'a'\n                                                                     │
│ (2) the list of words with more than five letters\n                                                             │
│ (3) the length of the longest word\n                                                                            │
│ (4) the count of palindromic words\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a_words = [w for w in words if w.startswith('a')]\n                                                         │
│     long_words = [w for w in words if len(w) > 5]\n                                                             │
│     longest_len = max(len(w) for w in words)\n                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DEST = {a} (anchor)  TASK span (79, 88)  ·  pivot=['    ']  ·  L=143 ──────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ (1) the list of words that start with 'a'\n                                                                     │
│ (2) the list of words with more than five letters\n                                                             │
│ (3) the length of the longest word\n                                                                            │
│ (4) the count of palindromic words\n                                                                            │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     a_words = [w for w in words if w.startswith('a')]\n                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean {a} (expect writes b) : 
 long_words = [w for w in words if len(w) > 5]
    longest_word_length = max(len(w) for w in words) if

inject q(b,c) (expect d)   : 
 palindromes = [w for w in words if w == w[::-1]]
    return a_words, palindromes, sum(1 for w in words if w == w[::-1]), max((len(w) for w in words), default=0)
</code>

inject q_reflect (expect ≠d) : 
 long_words = [w for w in words if len(w) > 5]
    long_words = [w for w in words if len(w) > 5]
    long_words = [w for w in words if len(w) > 5]
    longest_word_length = max(len(w) for w

## B1 · Query linearity (geometric)

Is the done-set encoded additively in query space? Cosine, over the concatenated GOAL q-head slices,
between `Δq(b,c)` and `Δq(b) + Δq(c)`  (each `Δq(X) = q({a}∪X) − q({a})`, measured against the anchor baseline).
**Predict:** high cosine ⇒ subtraction is linear/compositional over done-items.

In [7]:
pc = build_prompt(["c"])
qc = cache_src_proj(pc, [pivot_of(pc)], Q_LAYERS, "q")

def goal_q_vec(qcache):
    parts = [qcache[l][0, h * hd:(h + 1) * hd] for (l, h) in GOAL_HEADS if l in qcache]
    return torch.cat(parts)

# Δq(X) = q({a}∪X) − q({a})   (each done-update measured against the anchor baseline)
dq_b  = goal_q_vec(q_lincomb([(1.0, qb),  (-1.0, q0)]))
dq_c  = goal_q_vec(q_lincomb([(1.0, qc),  (-1.0, q0)]))
dq_bc = goal_q_vec(q_lincomb([(1.0, qbc), (-1.0, q0)]))
print(f"cos( Δq(b,c) , Δq(b)+Δq(c) )      = {cos(dq_bc, dq_b + dq_c):+.3f}")
print(f"cos( Δq(b,c) , Δq(b) )            = {cos(dq_bc, dq_b):+.3f}   (single-term baseline)")
print(f"cos( Δq(b)   , Δq(c) )            = {cos(dq_b,  dq_c):+.3f}   (are the two updates ~orthogonal?)")

cos( Δq(b,c) , Δq(b)+Δq(c) )      = +0.816
cos( Δq(b,c) , Δq(b) )            = +0.564   (single-term baseline)
cos( Δq(b)   , Δq(c) )            = +0.465   (are the two updates ~orthogonal?)


## B2 · Compositional causal injection  (the payload)

Inject a *synthesised* query `q_synth = qb + qc − q0`  (`q0 = q({a})`, `qb = q({a,b})`, `qc = q({a,c})`)
at the `{a}`-baseline pivot and generate.
**Predict:** writes item **d** only — the composed done-set behaves as {a,b,c}. Controls: real `qbc = q({a,b,c})`
should match; single terms narrow less. **Saturation** `qb+qc+qd−2·q0` → {a,b,c,d} all done → **boundary/return**.

*Note:* injection target is the fixed-geometry `{a}`-baseline prompt; only the position-free pre-RoPE queries are combined.

In [8]:
pd = build_prompt(["d"])
qd = cache_src_proj(pd, [pivot_of(pd)], Q_LAYERS, "q")
q_synth = q_lincomb([(1.0, qb), (1.0, qc), (-1.0, q0)])                    # ≈ done {a,b,c} -> expect d
q_sat   = q_lincomb([(1.0, qb), (1.0, qc), (1.0, qd), (-2.0, q0)])         # done {a,b,c,d} -> boundary

def gen_inject(qcache, tag, color):
    pf, dc = q_inject_specs(qcache, piv0)
    rich_print(f"[{color}]{tag}:[/{color}] "
               f"\n{escape(greedy(dest_tok, prefill_specs=pf, decode_specs=dc, n_steer=64, max_new=64))}")

rich_print(f"[grey70]clean {{a}}              (expect b) :[/grey70] \n{escape(greedy(dest_tok, max_new=64))}")
gen_inject(qbc,     "inject q(b,c) real (expect d) ", "sea_green3")
gen_inject(q_synth, "inject q_synth = b+c−q0 (expect d)", "gold3")
gen_inject(qb,      "inject q({b})       (expect c/d)", "grey58")
gen_inject(q_sat,   "inject q_sat = b+c+d−2q0 (→bound)", "salmon1")

clean {a}              (expect b) : 
 long_words = [w for w in words if len(w) > 5]
    longest_word_length = max(len(w) for w in words) if words else 0
    palindromic_count = sum(1 for w in words if w == w[::-1])
    return a_words, long

inject q(b,c) real (expect d) : 
 palindromes = [w for w in words if w == w[::-1]]
    return a_words, palindromes, sum(1 for w in words if w == w[::-1]), sum(1 for w in words if w == w[::-1])

# Example usage:
# words = ["

inject q_synth = b+c−q0 (expect d): 
 long_words = [w for w in words if len(w) > 5]
    longest_word_length = max((len(w) for w in words), default=0)
    palindromic_count = sum(1 for w in words if w == w[::-1])
    return a_words, long_words

inject q({b})       (expect c/d): 
 long_words = [w for w in words]
    max_length = max(len(w) for w in words) if words else 0
    longest_word = max(words, key=len) if words else ""
    palindromic_words = [w for w in words]
    longest_palindrome = max(len

inject q_sat = b+c+d−2q0 (→bound): 
 long_words = [w for w in words if len(w) == max(len(w) for w in words)]
    longest_word_length = max(len(w) for w in words)
    palindromic_words = [w for w in words if w == w[::-1]]
    return a_words, longest_word

## B2b · De-overlap correction (Gram–Schmidt)  —  is the shared component double-counted?

`Δb = qb−q0` and `Δc = qc−q0`  (`q0 = q({a})`) are **not orthogonal** (`cos(Δb,Δc) ≠ 0`), so the plain
synth `q0 + Δb + Δc` counts their shared component `s` **twice**. Since the real `Δbc = q({a,b,c})−q0`
is **already cached**, we settle it directly — no theory:

- **add:** `Δb + Δc`  (current B2)
- **GS :** `Δb + Δc − proj_b(Δc)`  — subtract the *shared* part (the **projection**, computed per layer), keep `−q0`.

Compare each reconstruction's `cos(·, Δbc)` and residual. **If `add` already tracks `Δbc`** (strong B1
additivity) the model double-counts nothing → GS would *over-correct*. **If `GS` is closer**, the
overlap is real and the GS-corrected query is the better payload. Then inject all three and read behavior.

In [9]:
# ── B2b: measure which reconstruction matches the real Δ(b,c), then inject the GS-corrected query ──
import torch
import torch.nn.functional as F

def _flat(d): return torch.cat([d[l].flatten() for l in Q_LAYERS])
Delta_bc = _flat(qbc) - _flat(q0)                       # ground-truth "b,c done" update (vs anchor baseline)

add_parts, gs_parts, cos_bc = [], [], []
q_synth_gs = {}                                          # {layer: (1, width)} for injection
for l in Q_LAYERS:
    db = (qb[l] - q0[l])                                 # (1, width)
    dc = (qc[l] - q0[l])
    fb, fc = db.flatten(), dc.flatten()
    proj = (fb @ fc) / (fb @ fb) * db                    # shared component of Δc along Δb (per layer)
    add_parts.append((db + dc).flatten())
    gs_parts.append((db + dc - proj).flatten())
    cos_bc.append((fb @ fc) / (fb.norm() * fc.norm()))
    q_synth_gs[l] = q0[l] + db + dc - proj               # = q0 + Δb + Δc_perp   (keeps −q0)

add_recon = torch.cat(add_parts)
gs_recon  = torch.cat(gs_parts)
rich_print(f"[grey70]mean per-layer cos(Δb, Δc) = {torch.stack(cos_bc).mean():.3f}   "
           f"(0 ⇒ orthogonal, no double-count)[/grey70]")
for name, v in [("add  Δb+Δc", add_recon), ("GS   Δb+Δc−proj_bΔc", gs_recon)]:
    rich_print(f"  {name:22s}  cos(·,Δbc)={F.cosine_similarity(v, Delta_bc, dim=0).item():+.3f}   "
               f"‖resid‖={(v - Delta_bc).norm():.2f}")

# ── behavioral read: inject add / GS / real, all via gen_inject from B2 ──
rich_print(f"[grey70]clean {{a}}                    (expect b) :[/grey70] \n{escape(greedy(dest_tok, max_new=64))}")
gen_inject(q_synth,    "inject q_synth    = b+c−q0           (expect d)", "gold3")
gen_inject(q_synth_gs, "inject q_synth_gs = b+c−q0−proj_bΔc  (expect d)", "cyan")
gen_inject(qbc,        "inject q(b,c) real                  (expect d)", "sea_green3")

mean per-layer cos(Δb, Δc) = 0.394   (0 ⇒ orthogonal, no double-count)

add  Δb+Δc              cos(·,Δbc)=+0.804   ‖resid‖=261.29

GS   Δb+Δc−proj_bΔc     cos(·,Δbc)=+0.825   ‖resid‖=193.62

clean {a}                    (expect b) : 
 long_words = [w for w in words if len(w) > 5]
    longest_word_length = max(len(w) for w in words) if words else 0
    palindromic_count = sum(1 for w in words if w == w[::-1])
    return a_words, long

inject q_synth    = b+c−q0           (expect d): 
 long_words = [w for w in words if len(w) > 5]
    longest_word_length = max((len(w) for w in words), default=0)
    palindromic_count = sum(1 for w in words if w == w[::-1])
    return a_words, long_words

inject q_synth_gs = b+c−q0−proj_bΔc  (expect d): 
 long_words = [w for w in words if w == w[::-1]]
    longest_word_length = max((len(w) for w in words), default=0)
    palindromic_count = sum(1 for w in words if w == w[::-1])
    return a_words, long_words,

inject q(b,c) real                  (expect d): 
 palindromes = [w for w in words if w == w[::-1]]
    return a_words, palindromes, sum(1 for w in words if w == w[::-1]), sum(1 for w in words if w == w[::-1])

# Example usage:
# words = ["